# Intro

In [1]:
import os, sys
import platform
import numpy as np
import pickle 

In [2]:
from collections import defaultdict

In [3]:
from dcms.models import DCMModel, DECMModel, ADECMModel, DWCMModel

## Load data

In [4]:
if platform.system() == 'Darwin':
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")


In [5]:
sys.path.insert(0, HOME)

In [6]:
DATA_FOLDER=HOME+'dati_elezioni/'

In [7]:
files=os.listdir(DATA_FOLDER)
files.sort()

In [8]:
files

['crisi_dicos.csv',
 'crisi_weighted_edgelist.csv',
 'ita_elections_dicos.csv',
 'ita_elections_weighted_edgelist.csv',
 'quirinale_dicos.csv',
 'quirinale_weighted_edgelist.csv']

In [9]:
l_dataset=len(files)//2

## Functions

In [10]:
def el2ks(el):
    all_nodes=np.concatenate((el['source_id'], el['target_id']))
    all_nodes=np.unique(all_nodes)
    k_out=np.zeros(len(all_nodes), dtype=int)
    k_in=np.zeros(len(all_nodes), dtype=int)
    s_out=np.zeros(len(all_nodes), dtype=int)
    s_in=np.zeros(len(all_nodes), dtype=int)
    node_index={node:i for i,node in enumerate(all_nodes)}
    for s,t,w in el:
        i_s=node_index[s]
        i_t=node_index[t]
        k_out[i_s]+=1
        k_in[i_t]+=1
        s_out[i_s]+=w
        s_in[i_t]+=w
    return k_out, k_in, s_out, s_in, all_nodes

# Case0: Crisis

In [11]:
i_file=0

In [12]:
el=np.genfromtxt(
    DATA_FOLDER+files[i_file+1],
    delimiter=',',
    skip_header=1,
    autostrip=True,
    dtype=[('source_id', '>U50'), ('target_id', '>U20'),('weight', 'i4')]
 )

In [13]:
dico=np.genfromtxt(
    DATA_FOLDER+files[i_file],
    delimiter=',',
    skip_header=1,
    autostrip=True,
    dtype=[('user_id', '>U50'), ('dico', '>U2'), ('h_dico', 'U2'), ('i_dico', 'U2')]
 )

In [14]:
dico_dict={}
bad_dicos=[]
for d in dico:
    if d['dico'].isnumeric():
        dico_dict[d['user_id']]=int(d['dico'])
    else:
        if d['dico'] not in bad_dicos:
            bad_dicos.append(d['dico'])
            print(d['dico'])

-1


In [50]:
cacca=np.unique(list(dico_dict.values()), return_counts=True)

In [51]:
np.vstack(cacca).T

array([[    0, 58832],
       [    1, 31874],
       [    2, 15168],
       [    3,  1304],
       [    4,  3637]])

In [16]:
n_nodes=np.concatenate((el['source_id'], el['target_id']))
n_nodes=np.unique(n_nodes)

In [17]:
len(n_nodes), len(dico_dict), len(dico_dict)/len(n_nodes)

(113876, 110815, 0.9731198847869612)

Who are the missing ones? I guees they are on smaller components

In [18]:
# Raggruppo in liste per efficienza, poi converto in array strutturati come el
_tmp = defaultdict(list)
for edge in el:
    src = edge['source_id'].strip()
    tgt = edge['target_id'].strip()
    d_src = dico_dict.get(src)
    if d_src is not None and d_src == dico_dict.get(tgt):
        _tmp[d_src].append(edge)

# defaultdict che restituisce array vuoti con lo stesso dtype di el
el_dico = defaultdict(
    lambda: np.empty(0, dtype=el.dtype),
    {k: np.array(v, dtype=el.dtype) for k, v in _tmp.items()}
 )

In [19]:
for key in el_dico.keys():
    print(key, len(el_dico[key]))

1 345876
0 322098
2 143937
3 1748
4 4578


In [20]:
del _tmp

## From the edgelist to the constraints

#### DiCo 1

In [21]:
aux=el2ks(el_dico[1])# DiCo 1

In [22]:
assert aux[0].sum()==aux[1].sum()==len(el_dico[1])

In [23]:
assert aux[2].sum()==aux[3].sum()

In [24]:
# Number of nodes, Number of edges, edge density
len(aux[4]), len(el_dico[1]), len(el_dico[1])/len(aux[4])**2

(31874, 345876, 0.0003404452594366783)

Ok, that's really challenging. So far, DCM Models were tested on datasets with 5k nodes. Here we have a little less than 32k nodes, and the number of links is a little more than 345k. Let's see if it works.

## DCM Models

### DWCM

#### Comparison

In [25]:
# with backend='pytorch'
with open(HOME + '/test/crisis_dwcm_old_theta.pkl', 'rb') as f:
    dwcm_old_theta = pickle.load(f)
with open(HOME + '/test/crisis_dwcm_old_gs.pkl', 'rb') as f:
    dwcm_old_gs = pickle.load(f)

In [26]:
# with backend='numba'
nprocs=1
with open(HOME + f'/test/crisis_dwcm_new_theta_nprocs_{nprocs}.pkl', 'rb') as f:
    dwcm_new_theta_theta_1 = pickle.load(f)
nprocs=8
with open(HOME + f'/test/crisis_dwcm_new_theta_nprocs_{nprocs}.pkl', 'rb') as f:
    dwcm_new_theta_theta_8 = pickle.load(f)

In [27]:
# with backend='numba'
nprocs=1
with open(HOME + f'/test/crisis_dwcm_new_gs_nprocs_{nprocs}.pkl', 'rb') as f:
    dwcm_new_gs_gs_1 = pickle.load(f)
nprocs=8
with open(HOME + f'/test/crisis_dwcm_new_gs_nprocs_{nprocs}.pkl', 'rb') as f:
    dwcm_new_gs_gs_8 = pickle.load(f)

In [28]:
for variant in ['theta', 'gs']:
    old = eval(f'dwcm_old_{variant}')
    print(f'{variant} pytorch: {old.sol.elapsed_time//60:} min, {old.sol.peak_ram_bytes//1024**2} MB')
    for nprocs in [1, 8]:
        new = eval(f'dwcm_new_{variant}_{variant}_{nprocs}')
        print(f'{variant} nprocs={nprocs}: {new.sol.elapsed_time//60:} min, {new.sol.peak_ram_bytes//1024**2} MB')

theta pytorch: 6.0 min, 791 MB
theta nprocs=1: 19.0 min, 0 MB
theta nprocs=8: 6.0 min, 46 MB
gs pytorch: 12.0 min, 1 MB
gs nprocs=1: 21.0 min, 117 MB
gs nprocs=8: 5.0 min, 240 MB


So, both in terms of i) calculation time and ii) RAM usage, numba wins on our old server. even in the case of the slower Gauss-Seidel with 8 processors the time is equivalent, but the RAM usage is reduced by  a factor  of 30!